# Healthcare Case Study
Companion notebook for the **Healthcare Case Study** section of Chapter 11.

Admin support (scheduling, insurance docs, referral status) and clinical support (summarizing records, finding labs) are held to very different standards. Neither one diagnoses, prescribes, or presents guesses as medical advice -- and a symptom red flag always breaks out of the normal flow.

This is the book's `route_healthcare_request`, unchanged:

In [1]:
def route_healthcare_request(req):
    if req["symptom_red_flag"]:
        return "urgent_human_escalation"
    if req["request_type"] == "admin":
        return "admin_agent_flow"
    if req["request_type"] == "clinical":
        return "clinician_review_required"
    return "manual_triage"

In [2]:
requests = [
    {"name": "move a follow-up appointment", "symptom_red_flag": False, "request_type": "admin"},
    {"name": "summarize recent labs for a nurse", "symptom_red_flag": False, "request_type": "clinical"},
    {"name": "patient reports chest pain", "symptom_red_flag": True, "request_type": "clinical"},
    {"name": "unclear request type", "symptom_red_flag": False, "request_type": "other"},
]

for req in requests:
    print(f"{route_healthcare_request(req):24s} <- {req['name']}")

admin_agent_flow         <- move a follow-up appointment
clinician_review_required <- summarize recent labs for a nurse
urgent_human_escalation  <- patient reports chest pain
manual_triage            <- unclear request type


Notice `symptom_red_flag` is checked **first**, before `request_type` -- even a request tagged "clinical" (which would otherwise just need clinician review) escalates urgently the moment a red-flag symptom shows up, regardless of what else is in the request.

**Privacy** is a core design rule alongside routing: minimum necessary access, role-based checks, and consent -- not just "can this agent technically read the record."

In [3]:
def can_access_record(agent_role: str, record_purpose: str, consent_on_file: bool) -> bool:
    if not consent_on_file:
        return False
    allowed_purposes = {
        "admin_agent": {"scheduling", "insurance_docs", "referral_status"},
        "clinical_support_agent": {"labs", "record_summary", "triage_note_draft"},
    }
    return record_purpose in allowed_purposes.get(agent_role, set())


print(can_access_record("admin_agent", "scheduling", consent_on_file=True))
print(can_access_record("admin_agent", "labs", consent_on_file=True))          # wrong role for this purpose
print(can_access_record("clinical_support_agent", "labs", consent_on_file=False))  # no consent on file

True
False
False
